# Benchmarking Stratified Patching

In [ ]:
import time
from collections.abc import Sequence
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray

from careamics.dataset_ng.patching_strategies import (
    PatchingStrategy,
    RandomPatchingStrategy,
    StratifiedPatchingStrategy,
)

In [ ]:
def demo_selected_patches(
    patching_strategy: PatchingStrategy,
    data_shapes: Sequence[Sequence[int]],
    epochs: int,
) -> tuple[Sequence[NDArray[np.int_]], NDArray[np.floating]]:
    """Create a map where all the patches have been selected from.

    Every time a patch is selected that area is incremented by 1.

    The time taken to generate each patch spec is also returned in a numpy array
    """
    tracking_arrays = [np.zeros(shape, dtype=int) for shape in data_shapes]
    n_patches = patching_strategy.n_patches
    patch_gen_times = np.zeros(epochs * n_patches, dtype=float)
    for epoch in range(epochs):
        for index in range(n_patches):
            t = time.time()
            patch_spec = patching_strategy.get_patch_spec(index)
            patch_gen_times[epoch*n_patches + index] = time.time() - t
            data_idx = patch_spec["data_idx"]
            sample_idx = patch_spec["sample_idx"]
            coord = patch_spec["coords"]
            patch_size = patch_spec["patch_size"]

            patch_slice = [
                slice(c, c + ps) for c, ps in zip(coord, patch_size, strict=True)
            ]
            tracking_arrays[data_idx][sample_idx, ..., *patch_slice] += 1
    return tracking_arrays, patch_gen_times

In [ ]:
def bench_patching_init(
    patching: Literal["stratif", "random"],
    data_shapes_sets: list[list[tuple[int, int, int, int]]],
    patch_size: tuple[int, int],
    repeats: int = 50,
):
    n_patches = np.zeros(len(data_shapes_sets), dtype=int)
    init_times = np.zeros((len(data_shapes_sets), repeats))
    for i, data_shapes in enumerate(data_shapes_sets):
        for r in range(repeats):
            t = time.time()
            patching_strat: StratifiedPatchingStrategy | RandomPatchingStrategy
            if patching == "stratif":
                patching_strat = StratifiedPatchingStrategy(
                    data_shapes, patch_size, seed=42
                )
            else:
                patching_strat = RandomPatchingStrategy(
                    data_shapes, patch_size, seed=42
                )
            init_times[i, r] = time.time() - t
            n_patches[i] = patching_strat.n_patches
    return init_times, n_patches

## Benchmarking initialization

We will compare the `StratifiedPatchingStrategy` to the `RandomPatchingStrategy` with
increasing number of patches. 

Additionally, we will see if increasing the number samples vs increasing the image size
makes a difference

In [ ]:
patch_size = (64, 64)
data_shapes_increase_samples = [
    [(1, 1, 2500, 1690)],
    [(2, 1, 2500, 1690)],
    [(4, 1, 2500, 1690)],
    [(8, 1, 2500, 1690)],
]
patch_size = (64, 64)
data_shapes_increase_size = [
    [(1, 1, 2500, 1690)],
    [(1, 1, 2500*2, 1690*2)],
    [(1, 1, 2500*3, 1690*3)],
    # [(1, 1, 2500*4, 1690*4)],
]

In [ ]:
stratif_init_times_samples, stratif_n_patches_samples = bench_patching_init(
    "stratif", data_shapes_increase_samples, patch_size
)
stratif_init_times_size, stratif_n_patches_size = bench_patching_init(
    "stratif", data_shapes_increase_size, patch_size
)

In [ ]:
random_init_times_samples, random_n_patches_samples = bench_patching_init(
    "random", data_shapes_increase_samples, patch_size
)
random_init_times_size, random_n_patches_size = bench_patching_init(
    "random", data_shapes_increase_size, patch_size
)

### Results

The results below show that the random patching strategy has almost no initialization 
overhead and performs with constant time relative to the number of patches.

In comparison the stratified patching strategy has some overhead for initialization with 
what seems to be linear time. However for almost 10,000 patches this about 0.35s. 
There does seem to be a small difference between whether there are more patches due to the 
number of samples or due to larger image sizes.

In [ ]:
stratif_order_samples = np.argsort(stratif_n_patches_samples)
random_order_samples = np.argsort(random_n_patches_samples)
stratif_order_size = np.argsort(stratif_n_patches_size)
random_order_size = np.argsort(random_n_patches_size)

plt.errorbar(
    stratif_n_patches_samples[stratif_order_samples],
    stratif_init_times_samples.mean(axis=1)[stratif_order_samples],
    yerr=stratif_init_times_samples.std(axis=1)[stratif_order_samples],
    label="stratified increasing samples",
)
plt.errorbar(
    random_n_patches_samples[random_order_samples],
    random_init_times_samples.mean(axis=1)[random_order_samples],
    yerr=random_init_times_samples.std(axis=1)[random_order_samples],
    label="random increasing samples",
)
plt.errorbar(
    stratif_n_patches_size[stratif_order_size],
    stratif_init_times_size.mean(axis=1)[stratif_order_size],
    yerr=stratif_init_times_size.std(axis=1)[stratif_order_size],
    label="stratified increasing size",
)
plt.errorbar(
    random_n_patches_size[random_order_size],
    random_init_times_size.mean(axis=1)[random_order_size],
    yerr=random_init_times_size.std(axis=1)[random_order_size],
    label="random increasing size",
)
plt.legend()
plt.ylabel("Patching initialization [s]")
plt.xlabel("N patches [#]")

## Benchmarking patch specs generation

In [ ]:
data_shapes = [(1, 1, 2500, 1690)]

In [ ]:
stratified_patching = StratifiedPatchingStrategy(data_shapes, patch_size, seed=42)
stratif_epochs_1, _ = demo_selected_patches(
    stratified_patching, data_shapes, epochs=1
)
stratif_epochs_200, stratif_gen_patch_times = demo_selected_patches(
    stratified_patching, data_shapes, epochs=200
)

In [ ]:
random_patching = RandomPatchingStrategy(data_shapes, patch_size, seed=42)
random_epochs_1, _ = demo_selected_patches(
    random_patching, data_shapes, epochs=1
)
random_epochs_200, random_gen_patch_times = demo_selected_patches(
    random_patching, data_shapes, epochs=200
)

### Results

The results show that the stratified patching strategy is about 3x slower than the random
patching strategy at generating patch specs. For 30,000 patches (roughly 30 epochs) 
the stratified patching will add about 1s of overhead whereas the random patching will 
add about 0.3s of overhead.

In [ ]:
stratif_mean = np.mean(stratif_gen_patch_times)
stratif_std = np.std(stratif_gen_patch_times)
stratif_total = np.sum(stratif_gen_patch_times)
stratif_n_patches = stratified_patching.n_patches
random_mean = np.mean(random_gen_patch_times)
random_std = np.std(random_gen_patch_times)
random_total = np.sum(random_gen_patch_times)
random_n_patches = random_patching.n_patches
print(
    "--- Stratified Patching\n"
    "Mean generation time:\n"
    f"1 patch: {stratif_mean:.2e} ± {stratif_std:.2e}s\n"
    f"30,000 patches: {stratif_mean*30000:.2f} ± {stratif_std*30000:.2f}s\n"
    f"200 epochs each with {stratif_n_patches} patches: {stratif_total:.2f}s\n"
    "--- Random Patching\n"
    "Mean generation time:\n"
    f"1 patch: {random_mean:.2e} ± {random_std:.2e}s\n"
    f"30,000 patches: {random_mean*30000:.2f} ± {random_std*30000:.2f}s\n"
    f"200 epochs each with {random_n_patches} patches: {random_total:.2f}s\n"
)

## Visual Comparison of patch distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(6, 8), constrained_layout=True)
axes[0, 0].imshow(stratif_epochs_1[0].squeeze())
axes[0, 1].imshow(stratif_epochs_200[0].squeeze())
axes[1, 0].imshow(random_epochs_1[0].squeeze())
axes[1, 1].imshow(random_epochs_200[0].squeeze())

axes[0, 0].set_ylabel("Stratified")
axes[1, 0].set_ylabel("Random")
axes[0, 0].set_title("1 epoch")
axes[0, 1].set_title("200 epochs")